In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

autodf = pd.read_csv('data/vehicles.csv')

# =============================================================================
# STEP 1: PREPROCESSING
# =============================================================================

# Drop columns that won't help regression
# (IDs, VINs are unique identifiers — no predictive signal)
df = autodf.drop(columns=['id', 'VIN'])
autodf.dropna(inplace=True)
autodf = autodf[autodf['price'] > 0] # remove cars with price of zero


In [ ]:
# --- Ordinal Encoding (order matters) ---
# These features have a meaningful numeric ranking,
# so we map them to integers rather than one-hot encoding.
# Example: 'good' < 'excellent' < 'like new' — the gap between values is intentional.
ordinal_map = {
    'condition': {'salvage': 0, 'fair': 1, 'good': 2, 'excellent': 3, 'like new': 4, 'new': 5},
    'cylinders': {'3 cylinders': 3, '4 cylinders': 4, '5 cylinders': 5,
                  '6 cylinders': 6, '8 cylinders': 8, '10 cylinders': 10, '12 cylinders': 12},
    'size':      {'sub-compact': 0, 'compact': 1, 'mid-size': 2, 'full-size': 3},
}
for col, mapping in ordinal_map.items():
    df[col] = df[col].map(mapping)
    # NOTE: Values not in the mapping become NaN — handled by fillna below

# --- One-Hot Encoding (no natural order) ---
# These features are purely categorical with no ranking.
# drop_first=True drops one dummy per group to avoid multicollinearity
# (the "reference" category is absorbed into the intercept).
nominal_cols = ['region', 'manufacturer', 'model', 'fuel',
                'title_status', 'transmission', 'drive',
                'type', 'paint_color', 'state']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)



In [ ]:
# --- Handle Nulls ---
# Lasso can't handle NaNs. Median imputation is robust to outliers
# (mean would be skewed by $200k exotics, for example).
df = df.fillna(df.median(numeric_only=True))

# --- Split features and target ---
X = df.drop(columns=['price'])
y = df['price']

# Optional but recommended: log-transform price
# Car prices are right-skewed (a few $200k cars distort the model).
# Log makes the distribution more normal, improving Lasso's fit.
# If you do this, remember: predictions will be in log-dollars → np.expm1() to reverse.
# y = np.log1p(y)

# =============================================================================
# STEP 2: TRAIN/TEST SPLIT
# =============================================================================

# Hold out 20% of data for honest evaluation after feature selection.
# IMPORTANT: fit scaler + Lasso only on X_train — never peek at test data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =============================================================================
# STEP 3: SCALE FEATURES
# =============================================================================

# WHY scaling is critical for Lasso:
# Lasso penalizes the *absolute size* of coefficients (L1 penalty: sum |β|).
# Without scaling, a feature measured in miles (0–300,000) gets a tiny raw
# coefficient compared to one measured in years (0–30) — even if they matter equally.
# StandardScaler makes every feature mean=0, std=1, so coefficients are comparable.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # apply same scale to test

# =============================================================================
# STEP 4: LASSO CV
# =============================================================================

# LassoCV automatically searches for the best regularization strength (alpha)
# via cross-validation — no manual tuning needed.
#
# HOW IT WORKS:
# Alpha controls the penalty on large coefficients:
#   - alpha → 0 : behaves like OLS (no shrinkage, keeps all features)
#   - alpha → ∞ : shrinks everything to zero (all features dropped)
# LassoCV tests many alphas and picks the one with the lowest CV error.
#
# Key hyperparameters:
#   cv=5        — 5-fold cross-validation (good default; use 10 for smaller datasets)
#   max_iter    — increase if you get "did not converge" warnings
#   n_alphas    — number of alpha candidates to try (default 100, usually fine)
lasso = LassoCV(cv=5, random_state=42, max_iter=10_000)
lasso.fit(X_train_scaled, y_train)

print(f"Best alpha chosen by CV: {lasso.alpha_:.4f}")
# Higher alpha = more features zeroed out (sparser, simpler model)
# Lower alpha  = more features retained (closer to plain OLS)

# =============================================================================
# STEP 5: EXTRACT & INTERPRET FEATURE IMPORTANCE
# =============================================================================

# Each coefficient tells you: "holding all else equal, a 1-std-deviation
# increase in this feature changes price by $X" (or log-$ if you log-transformed y).
# A coefficient of 0 means Lasso decided this feature adds no signal.
coef_df = pd.DataFrame({
    'feature':     X.columns,
    'coefficient': lasso.coef_
}).sort_values('coefficient', key=abs, ascending=False)

# Split into kept vs. dropped features
selected = coef_df[coef_df['coefficient'] != 0]
dropped  = coef_df[coef_df['coefficient'] == 0]

print(f"\nFeatures kept : {len(selected)} / {len(X.columns)}")
print(f"Features zeroed out (irrelevant): {len(dropped)}")
print("\nTop 20 most important features:")
print(selected.head(20).to_string(index=False))

# =============================================================================
# STEP 6: EVALUATE ON HELD-OUT TEST SET
# =============================================================================

from sklearn.metrics import mean_absolute_error, r2_score

y_pred = lasso.predict(X_test_scaled)

# If you log-transformed y above, reverse it here:
# y_pred = np.expm1(y_pred)
# y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f"\nTest MAE : ${mae:,.0f}")   # avg dollar error on unseen cars
print(f"Test R²  : {r2:.3f}")       # 1.0 = perfect, 0.0 = predicts the mean

# =============================================================================
# STEP 7: VISUALIZE (optional but useful)
# =============================================================================

top_n = 20
top = selected.head(top_n)

plt.figure(figsize=(9, 6))
colors = ['steelblue' if c > 0 else 'tomato' for c in top['coefficient']]
plt.barh(top['feature'][::-1], top['coefficient'][::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel("Lasso Coefficient (scaled units → price $)")
plt.title(f"Top {top_n} Features by |Coefficient| — LassoCV")
plt.tight_layout()
plt.show()
# Blue bars  → positively associated with higher price (e.g. low odometer, diesel)
# Red bars   → negatively associated with price      (e.g. salvage title, high miles)

# =============================================================================
# STEP 8: USE SELECTED FEATURES FOR A DOWNSTREAM MODEL
# =============================================================================

# The real payoff: use Lasso's selections to trim your feature matrix
# before feeding into a more powerful model (XGBoost, Random Forest, etc.)
selected_features = selected['feature'].tolist()

X_train_selected = X_train[selected_features]
X_test_selected  = X_test[selected_features]

print(f"\nReduced from {X.shape[1]} → {len(selected_features)} features")
print("Ready to feed into your final model.")